# GeoCPU v47：基礎能力を維持する追加学習と全比較

**準備済みの候補です。新しいGPU学習・モデル得点は、このノートブックを実行するまで未測定です。**

既存T1〜T8の問題・採点・正式な生成条件を保持し、別枠のPython実用評価240件と知識診断を追加します。追加学習は保存済みstep30,500から新枝へ行います。Python512行と知識336行を追加し、旧教材と凍結した親モデルを使って基礎能力を保つ設計です。

実行順は **準備 → 元モデルの評価 → CPU事前確認 → 新枝で1,000更新 → 同条件評価 → 比較表**。各節を順に実行してください。今の評価が動いている場合、まず別のCPU Colabで `GeoCPU_live_monitor_20261107.ipynb` を使います。このノートブックは稼働中の仕事を自動停止しません。

GitHubの研究記録は `corcondor/geocpu-diagnostics` のv47用PRで管理します。詳細は同梱の `README_JA.md`、`TRAINING.md`、技術ブリーフを参照してください。


## 1. Driveと実行用ファイル


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, stat, sys, tempfile, zipfile

STUDY = Path('/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106')
ARCHIVE = STUDY / 'GeoCPU_practical_training_logging_20261107' / 'GeoCPU_practical_training_logging_20261107.zip'
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'成果物ZIPの場所をARCHIVEへ指定してください: {ARCHIVE}')

# 既存コードへ重ねず、毎回新しいローカル作業フォルダーへ展開します。
WORK = Path(tempfile.mkdtemp(prefix='geocpu_practical_', dir='/content'))
with zipfile.ZipFile(ARCHIVE) as z:
    names=z.namelist()
    if len(names)!=len(set(names)):
        raise ValueError('ZIPの重複パス')
    for m in z.infolist():
        if not (WORK/m.filename).resolve().is_relative_to(WORK.resolve()) or stat.S_ISLNK(m.external_attr >> 16):
            raise ValueError('ZIPの不正なパス')
    manifest=json.loads(z.read('PACKAGE_MANIFEST.json'))
    expected={r['path']:r for r in manifest['files']}
    actual={m.filename for m in z.infolist() if not m.is_dir()}-{'PACKAGE_MANIFEST.json'}
    if actual!=set(expected):
        raise ValueError('ZIPと固定manifestのファイル集合が不一致')
    for name, row in expected.items():
        raw=z.read(name)
        if len(raw)!=row['size_bytes'] or hashlib.sha256(raw).hexdigest()!=row['sha256']:
            raise ValueError(f'内容ハッシュ不一致: {name}')
    z.extractall(WORK)
sys.path.insert(0, str(WORK))
os.chdir(WORK)
print('作業フォルダー:', WORK)
print('固定ファイル検証:', len(expected), '件')


## 2. 比較条件を固定する

`FULL_KNOWLEDGE=False`でもT3のJamC-QA 2,309問、T4のJMMLU 7,536問を含む元T1〜T8はすべて対象です。`True`はさらに、その同じ知識問題で検索なし／検索ありの生成を各全件行います。**1チェックポイントにつき19,690生成を追加**するため、初回の小試行ではFalseとしています。前後で同じ値を使ってください。

基準モデルの完了済みT1〜T8は、保存原票とハッシュが一致する場合だけ再利用します。途中結果を完了扱いにはしません。


In [ ]:
from datetime import datetime, timezone
from live_logging import run_with_logging
from train_candidate import refuse_active_work
import uuid

TAG = 'practical_20261107_v1'
VARIANT = 'multistep'  # 対照条件は answer_only または replay_only（別の新枝名になる）
STEPS = 1000
NEW_REPEAT = 16
FULL_KNOWLEDGE = False
BASE_RUN = 'v47_from28000_to30500_curriculum_v1'
BASE_STEP = 30500
CANDIDATE_RUN = f'{TAG}_{VARIANT}_{BASE_STEP+STEPS}'
ORIGINAL_BASELINE = STUDY / 'v47_train_eval_suite_step_30500' / 'original_tiers'
BASE_RESULTS = STUDY / f'{TAG}_baseline_{"full_knowledge" if FULL_KNOWLEDGE else "pilot"}'
CANDIDATE_RESULTS = STUDY / f'{CANDIDATE_RUN}_eval_{"full_knowledge" if FULL_KNOWLEDGE else "pilot"}'
CONSOLE_ROOT = STUDY / f'{TAG}_console'
# 親データの正確なキャッシュがある場合だけ指定。省略時は元の再構成を使います。
PREPARED_PARENT = None

def run_logged(args, label, watch_root=None):
    out=CONSOLE_ROOT / (label+'_'+uuid.uuid4().hex[:8])
    rc=run_with_logging([sys.executable,'-u',*map(str,args)], out, watch_root=watch_root, cwd=WORK)
    if rc:
        raise RuntimeError(f'{label}が終了コード{rc}で停止しました。ログ: {out}')
    return out

def training_request():
    request=[WORK/'train_candidate.py','--study',STUDY,'--parent-run',BASE_RUN,
             '--parent-step',BASE_STEP,'--output',STUDY/CANDIDATE_RUN,
             '--variant',VARIANT,'--steps',STEPS,'--new-repeat',NEW_REPEAT]
    if PREPARED_PARENT is not None:
        request.extend(['--prepared-parent',Path(PREPARED_PARENT)])
    return request

print('親:', BASE_RUN, BASE_STEP)
print('新枝:', CANDIDATE_RUN)
print('生成による全件知識追加比較:', FULL_KNOWLEDGE)


## 3. CPUで問題集合・実行計画を確認


In [ ]:
import frozen_eval
from eval_candidate import make_plan
print(frozen_eval.verify_frozen())
plan=make_plan(FULL_KNOWLEDGE)
for row in plan['plan']:
    print(row)
print('このセルではモデルをロードしていません。')


## 4. 基準モデルを評価する（空いているGPUで実行）

完了済みT1〜T8を検証して新しい結果フォルダーへコピーし、補助試験・追加試験を評価します。元の保存結果は変更しません。元T1〜T8の記録が不足している場合は停止します。そのときは結果の不足理由を確認してください。

新補助評価は一問ごとの原回答と採点を保存します。セルが長時間静かでも5秒ごとに生存確認を表示します。再実行時は同一条件の採点済み問題を二重生成せず再開します。異常終了したロックは自動消去しません。


In [ ]:
refuse_active_work(STUDY)
args=[WORK/'eval_candidate.py','run','--study',STUDY,'--run',BASE_RUN,'--step',BASE_STEP,
      '--output',BASE_RESULTS,'--reuse-original-from',ORIGINAL_BASELINE]
if BASE_RESULTS.exists(): args.append('--resume')
if FULL_KNOWLEDGE: args.append('--full-knowledge')
run_logged(args,'baseline',watch_root=BASE_RESULTS)


## 5. 追加学習のCPU事前確認

旧422,500行のハッシュ、追加教材の語彙・観測マスク、元のsamplerによる1,000更新分の提示計画を確認します。**旧教材の教師トークン75%以上、新教材1,000回以上の提示**を満たさない場合は学習を開始しません（replay_onlyは新教材提示の条件を除く）。

この事前確認は、この成果物を作った環境では未実行です。旧全教材のキャッシュ、または原本Grand ZIPとbridge教材などの再構成依存が必要です。不足を別データで埋めて先へ進めることはしません。


In [ ]:
train_args=training_request()
preflight_path=CONSOLE_ROOT/('preflight_'+uuid.uuid4().hex[:8]+'.json')
run_logged([*train_args,'--preflight-only','--preflight-report',preflight_path],'preflight')
preflight=json.loads(preflight_path.read_text())
validated_train_args=list(map(str,train_args))
print(json.dumps(preflight['preview'],ensure_ascii=False,indent=2))


## 6. 新枝で追加学習する

このセルはGPU学習を起動します。親と同じ単一の `NVIDIA A100-SXM4-40GB` が必要です。前の事前確認が通った条件で実行します。親モデル・元教材・実行中ジョブは変更しません。

損失・KL・学習率・教材別提示数・実際のトークン数・VRAM・経過時間を10更新ごとに記録します。低い学習損失は、試験の正答率向上の証拠とは扱いません。


In [ ]:
if not preflight_path.is_file():
    raise RuntimeError('先にCPU事前確認を実行してください。')
if (preflight.get('variant')!=VARIANT or preflight.get('steps')!=STEPS
        or validated_train_args!=list(map(str,training_request()))):
    raise RuntimeError('事前確認後に親・新枝・提示倍率などの設定が変わっています。CPU事前確認をやり直してください。')
run_logged(train_args,'training')
saved=STUDY/CANDIDATE_RUN/'training_saved.json'
print(saved.read_text())


## 7. 新しい保存モデルを同条件で評価する

元T1〜T8は原評価器で実行し、補助・追加評価は基準モデルと同じ問題・検索コーパス・生成予算で実行します。元T1〜T8は元実装の表示間隔を保ち、補助評価は一問ごとに更新します。計測エラー・未計測は0点にしません。


In [ ]:
refuse_active_work(STUDY)
args=[WORK/'eval_candidate.py','run','--study',STUDY,'--run',CANDIDATE_RUN,
      '--step',BASE_STEP+STEPS,'--output',CANDIDATE_RESULTS]
if CANDIDATE_RESULTS.exists(): args.append('--resume')
if FULL_KNOWLEDGE: args.append('--full-knowledge')
run_logged(args,'candidate_evaluation',watch_root=CANDIDATE_RESULTS)


## 8. 全比較表と基礎能力の保持判定


In [ ]:
from compare_results import write_reports
from IPython.display import display, Markdown
REPORT_DIR=STUDY/(CANDIDATE_RUN+'_comparison_'+uuid.uuid4().hex[:8])
comparison=write_reports(BASE_RESULTS,CANDIDATE_RESULTS,REPORT_DIR)
display(Markdown((REPORT_DIR/'comparison.md').read_text()))
print('比較CSV:',REPORT_DIR/'comparison.csv')
print('原票付きJSON:',REPORT_DIR/'comparison.json')


## 判定と次の試行

最初の目安は、T1/T2/T5とT4コード状態追跡の低下が各2ポイント以内で、Python120の最終回答が5ポイント以上、または元JMMLU/JamC-QAが2ポイント以上改善することです。これは工学的な試行基準で、統計的有意差や一般能力の保証ではありません。T6/T7/T8と生の失敗例も確認します。結果が不足している場合は判定を保留し、チェックポイントを自動採用しません。

多段の教師信号が効いたかを調べる対照は、同じ親から `VARIANT='answer_only'` と `VARIANT='replay_only'` を各1,000更新します。設定を変えたら新しい出力先を使い、事前確認から実施します。各条件の実際の教師トークン数・提示数も比較してください。

研究記録には、コードcommit、各入力ハッシュ、完了率、点数、退行した問題、未測定・例外を残します。今回のPRに実験結果を追記し、次の変更は別commitで比較できるようにします。
